# BERT finetune

In [161]:
from transformers import BertTokenizer, BertModel
import torch
import numpy as np
import pandas as pd
from torchsummaryX import summary
import torch.nn as nn
import os
import sys
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

## BertModel analysis

In [49]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')  
model = BertModel.from_pretrained('bert-base-uncased')

In [72]:
# print the model architecture
print(model)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [61]:
for name, module in model.named_modules():
    print(name)


embeddings
embeddings.word_embeddings
embeddings.position_embeddings
embeddings.token_type_embeddings
embeddings.LayerNorm
embeddings.dropout
encoder
encoder.layer
encoder.layer.0
encoder.layer.0.attention
encoder.layer.0.attention.self
encoder.layer.0.attention.self.query
encoder.layer.0.attention.self.key
encoder.layer.0.attention.self.value
encoder.layer.0.attention.self.dropout
encoder.layer.0.attention.output
encoder.layer.0.attention.output.dense
encoder.layer.0.attention.output.LayerNorm
encoder.layer.0.attention.output.dropout
encoder.layer.0.intermediate
encoder.layer.0.intermediate.dense
encoder.layer.0.intermediate.intermediate_act_fn
encoder.layer.0.output
encoder.layer.0.output.dense
encoder.layer.0.output.LayerNorm
encoder.layer.0.output.dropout
encoder.layer.1
encoder.layer.1.attention
encoder.layer.1.attention.self
encoder.layer.1.attention.self.query
encoder.layer.1.attention.self.key
encoder.layer.1.attention.self.value
encoder.layer.1.attention.self.dropout
encoder.

In [62]:
class CustomBERTModel(nn.Module):
    def __init__(self, bert_model):
        super(CustomBERTModel, self).__init__()
        self.bert_model = bert_model
    
    def forward(self, input_ids, token_type_ids, attention_mask):
        outputs = self.bert_model(input_ids=input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
        return outputs.pooler_output, outputs.last_hidden_state

In [71]:
# create a sample input, with batch size 2 
input_text = ["Hello, my dog is cute", "Hello, my cat is cute too."]
input = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True)
print(input)
custom_model = CustomBERTModel(model)

output = custom_model(input["input_ids"], input["token_type_ids"], input["attention_mask"])
output_pooler, output_last_hidden_state = output
output_pooler.shape, output_last_hidden_state.shape

{'input_ids': tensor([[  101,  7592,  1010,  2026,  3899,  2003, 10140,   102,     0,     0],
        [  101,  7592,  1010,  2026,  4937,  2003, 10140,  2205,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


(torch.Size([2, 768]), torch.Size([2, 10, 768]))

## dataset Imdb

sentiment analysis, classification task with label "positive" and "negative"

In [ ]:
from d2l import torch as d2l

d2l.DATA_HUB['aclImdb'] = (d2l.DATA_URL + 'aclImdb_v1.tar.gz',
                          '01ada507287d82875905620988597833ad4e0903')

data_dir = d2l.download_extract('aclImdb', 'aclImdb')


In [86]:
# read dataset from aclImdb
def read_imdb(data_dir, is_train):
    data, labels = [], []
    for label in ('pos', 'neg'):
        folder_name = os.path.join(data_dir, 'train' if is_train else 'test', label)
        for file in os.listdir(folder_name):
            with open(os.path.join(folder_name, file), 'rb') as f:
                review = f.read().decode('utf-8').replace('\n', '')
                data.append(review)
                labels.append(1 if label == 'pos' else 0)
    return data, labels

In [87]:
class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, data, labels, max_len=512, tokenizer=tokenizer):
        self.data = data
        self.labels = labels
        self.max_len = max_len
        self.tokenizer = tokenizer
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        text = self.data[idx]
        tokens = self.tokenizer(text, max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt")
        return tokens, self.labels[idx]

In [122]:
# load the data
def load_imdb(data_dir, batch_size=32, max_len=512, tokenizer=tokenizer):
    train_data = read_imdb(data_dir, is_train=True)
    test_data = read_imdb(data_dir, is_train=False)
    # create the dataset
    train_dataset = IMDbDataset(train_data[0], train_data[1], max_len=max_len, tokenizer=tokenizer)
    test_dataset = IMDbDataset(test_data[0], test_data[1], max_len=max_len, tokenizer=tokenizer)
    # create the dataloader (iterator)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size)
    
    return train_loader, test_loader, train_dataset, test_dataset


In [123]:
train_loader, test_loader, train_dataset, test_dataset = load_imdb(
    data_dir, batch_size=32, max_len=512, tokenizer=tokenizer
)

In [118]:
train_loader.dataset[0][0].keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])

In [152]:
print(
    "train_dataset length: {}, batch_size: {}, train_loader length: {}".format(
        train_dataset.__len__(), train_loader.batch_size, train_loader.__len__()
    )
)
# i wanna revert the tokenized input back to text
print("tokens of first sample in train_dataset: ", tokenizer.convert_ids_to_tokens(train_dataset[3][0]["input_ids"].numpy().squeeze()))


train_dataset length: 25000, batch_size: 32, train_loader length: 782
tokens of first sample in train_dataset:  ['[CLS]', 'summary', '-', 'this', 'game', 'is', 'the', 'best', 'spider', '-', 'man', 'to', 'hit', 'the', 'market', '!', 'you', 'fight', 'old', 'foe', '##s', 'such', 'as', 'scorpion', ',', 'rhino', ',', 'venom', ',', 'doctor', 'octopus', ',', 'carnage', ',', '.', '.', '.', 'and', 'exclusive', 'to', 'the', 'game', '.', '.', '.', 'monster', '-', 'o', '##ck', '!', 'monster', '-', 'o', '##ck', 'is', 'the', 'sy', '##mb', '##iot', '##e', 'carnage', 'on', 'dock', 'o', '##ck', "'", 's', 'body', '.', '<', 'br', '/', '>', '<', 'br', '/', '>', 'storyline', '-', 'dock', 'o', '##ck', 'was', 'supposedly', 'reformed', 'and', 'using', 'his', 'inventions', 'for', 'mankind', '.', '.', '.', 'supposedly', '.', '.', '.', 'he', 'was', 'really', 'plan', '##ing', 'a', 'sy', '##mb', '##iot', '##e', 'invasion', '!', 'see', 'the', 'rest', 'for', 'yourself', '.', '<', 'br', '/', '>', '<', 'br', '/', '>',

## fine tune, sentiment analysis task, classification

In [110]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
train_loader, test_loader = load_imdb(data_dir, batch_size=32, max_len=512, tokenizer=tokenizer)


In [154]:
class BERTFinetuneClassifier(nn.Module):
    def __init__(self, bert_model, num_classes):
        super(BERTFinetuneClassifier, self).__init__()
        self.bert_model = bert_model
        self.classifier = nn.Linear(bert_model.config.hidden_size, num_classes)
        self.softmax = nn.Softmax(dim=1)
    
    def forward(self, input_ids, token_type_ids, attention_mask):
        outputs = self.bert_model(input_ids=input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state
        # we only want the hidden state of the [CLS] token, which is the first token
        cls_hidden_state = last_hidden_state[:, 0, :]
        logits = self.classifier(cls_hidden_state) # (batch_size, num_classes)
        probs = self.softmax(logits) # (batch_size, num_classes)
        return probs

In [153]:
text = "Hello, my dog is cute"
tokens = tokenizer(text, return_tensors="pt")
# print(tokens)
output = model(tokens.input_ids, tokens.token_type_ids, tokens.attention_mask)
output.last_hidden_state[:, 0, :].shape

torch.Size([1, 768])

In [157]:
bert_finetune_classifier = BERTFinetuneClassifier(model, num_classes=2)
probs = bert_finetune_classifier(tokens.input_ids, tokens.token_type_ids, tokens.attention_mask)
probs.shape, probs

(torch.Size([1, 2]), tensor([[0.6235, 0.3765]], grad_fn=<SoftmaxBackward0>))

In [160]:
for i, (tokens, label) in enumerate(train_loader):
    print(tokens["input_ids"].shape, tokens["token_type_ids"].shape, tokens["attention_mask"].shape, label.shape)
    break

torch.Size([32, 1, 512]) torch.Size([32, 1, 512]) torch.Size([32, 1, 512]) torch.Size([32])


## training

In [171]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

def train_bert(model, train_loader, test_loader, num_epochs=5, 
               lr=1e-4, device = "cuda" if torch.cuda.is_available() else "cpu"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        for i, (batch, labels) in enumerate(train_loader):
            try:
                # move the batch to device
                input_ids = batch["input_ids"].squeeze(1).to(device)
                token_type_ids = batch["token_type_ids"].squeeze(1).to(device)
                attention_mask = batch["attention_mask"].squeeze(1).to(device)
                labels = labels.to(device)
                # forward pass
                output = model(input_ids, token_type_ids, attention_mask)
                loss = criterion(output, labels)
                # backward pass
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                # draw training process
                print(f"Epoch {epoch}, batch {i}, loss: {loss.item()}")
                train_loss += loss.item()
            except RuntimeError as e:
                print(f"Error at epoch {epoch}, batch {i}: {e}")
                if "CUDA out of memory" in str(e):
                    print("Out of memory error. Skipping batch.")
                    torch.cuda.empty_cache()
                else:
                    raise e
        # print(f"Epoch {epoch}, avg loss: {train_loss/len(train_loader)}")
        
        # # Evaluate on test set
        # model.eval()
        # test_accuracy = 0
        # with torch.no_grad():
        #     for batch, labels in test_loader:
        #         input_ids = batch["input_ids"].squeeze(1).to(device)
        #         token_type_ids = batch["token_type_ids"].squeeze(1).to(device)
        #         attention_mask = batch["attention_mask"].squeeze(1).to(device)
        #         labels = labels.to(device)
        #         output = model(input_ids, token_type_ids, attention_mask)
        #         test_accuracy += accuracy_score(labels.cpu(), output.argmax(1).cpu())
        # test_accuracy /= len(test_loader)
        # print(f"Epoch {epoch}, test accuracy: {test_accuracy}")


In [172]:
bert_finetune_classifier = BERTFinetuneClassifier(model, num_classes=2)
train_bert(bert_finetune_classifier, train_loader, test_loader, num_epochs=1)

RuntimeError: CUDA error: unspecified launch failure
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


# draft

In [1]:
from d2l import torch as d2l

In [ ]:
d2l.tokenize()
d2l.Vocab()

In [1]:
a = [1,2,3,4]
b = [6,7,8,9]
from torch.utils.data import DataLoader
dataset = list(zip(a, b))
dataloader = DataLoader(dataset, batch_size=2)

In [14]:
for i in dataloader:
    print(i)
    

[tensor([1, 2]), tensor([6, 7])]
[tensor([3, 4]), tensor([8, 9])]


In [12]:
len(dataloader)

2

In [ ]:
bert = d2l.BERTModel